# 03 - Modeling

Rubric: *Modeling (5 pts)* -- several models, tuning, and an advanced model.

Splits are chronological. Never `train_test_split(shuffle=True)` on time series.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from smaz import backtest, utils
from algotrade import ingest, model, strategy, transform

utils.set_plot_defaults()
pd.set_option("display.max_columns", 80)

In [ ]:
df = pd.read_parquet(transform.PROCESSED / "dataset.parquet")
feats = transform.feature_columns(df)
target = transform.TARGET

labelled = df.dropna(subset=[target])
split = model.time_split(labelled, valid_start="2021-01-01", test_start="2023-01-01")
{k: len(getattr(split, k)) for k in ("train", "valid", "test")}

## Compare models

In [ ]:
results = {}
for kind in ["decision_tree", "random_forest", "logistic", "xgboost"]:
    clf = model.fit_classifier(split.train, feats, target, kind=kind)
    results[kind] = model.evaluate(clf, split.valid, feats, target)

pd.DataFrame(results)

## Hyperparameter tuning

Worth a point. Use `TimeSeriesSplit`, not `KFold`.

In [ ]:
# from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV


## Feature importance

In [ ]:
# clf = model.fit_classifier(split.train, feats, target, kind="xgboost")
# pd.Series(clf.feature_importances_, index=feats).sort_values(ascending=False).head(25)